In [1]:
import json
import hashlib
import sys
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3.5:9b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"

# Archivos compartidos en 05_Pruebas
with open("../casos_comparacion_v1.json", "rb") as f:
    datos_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    datos_config = f.read()

casos = json.loads(datos_casos)
configuracion = json.loads(datos_config)

# Verificar integridad
sha_casos = hashlib.sha256(datos_casos).hexdigest()
sha_config = hashlib.sha256(datos_config).hexdigest()

assert sha_casos == configuracion["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert configuracion["api_url"] == API_LOCAL + "/api/chat"

# Conexión directa al servidor local
cliente_local = build_opener(ProxyHandler({}))

with cliente_local.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

with cliente_local.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos = json.load(r)["models"]

modelo_info = next(
    (m for m in modelos if m["name"] == MODELO),
    None
)

assert modelo_info is not None, "Qwen3.5 no aparece instalado."

# Consultar las opciones de razonamiento anunciadas
peticion = Request(
    API_LOCAL + "/api/show",
    data=json.dumps({"model": MODELO}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

with cliente_local.open(peticion, timeout=15) as r:
    detalles = json.load(r)

print("Python:", sys.version.split()[0])
print("Ollama:", version_ollama)
print("Modelo:", MODELO)
print("Identificador:", modelo_info["digest"][:12])
print("Casos verificados:", len(casos))
print("SHA256 casos:", sha_casos)
print("Prompt:", configuracion["prompt_version"])
print("Razonamiento disponible:", detalles.get("capabilities"))
print("Controles thinking:", detalles.get("thinking", "No informados"))
print("\nVerificación completada")


Python: 3.11.16
Ollama: 0.40.13
Modelo: qwen3.5:9b-q4_K_M
Identificador: 56671c2ab938
Casos verificados: 6
SHA256 casos: f73d159bb43cc9c93d2768e997052f4381a2c24bb046c82826b1793c1b8da37d
Prompt: V3
Razonamiento disponible: ['completion', 'vision', 'tools', 'thinking']
Controles thinking: {'values': [False, True], 'default': True}

Verificación completada


In [2]:
import copy
import json
import time
from urllib.request import Request

# Caso diagnóstico independiente de S-005 a S-010
contenido = (
    "Autores: Autor X y Autor Y.\n"
    "Predicción recibida: favorable (simulada).\n"
    "Modelo predictor y puntuación: no disponibles.\n"
    "[E1] Autor X tiene un artículo sobre bases de datos académicas.\n"
    "[E2] Autor Y tiene un artículo sobre recuperación de información.\n"
    "No se proporcionan datos de colaboración previa."
)

# Copiar la configuración compartida sin modificarla
solicitud_q35 = copy.deepcopy(
    configuracion["request_defaults"]
)

solicitud_q35["model"] = MODELO
solicitud_q35["think"] = False

# Parámetros recomendados para Qwen3.5 sin razonamiento
solicitud_q35["options"].update({
    "num_ctx": 4096,
    "num_predict": 2048,
    "temperature": 0.7,
    "top_p": 0.8,
    "top_k": 20,
    "min_p": 0.0,
    "presence_penalty": 1.5,
    "repeat_penalty": 1.0
})

# Mantener exactamente el prompt V3
solicitud_q35["messages"] = [
    {
        "role": "system",
        "content": configuracion["system_prompt"]
    },
    {
        "role": "user",
        "content": configuracion["user_prompt_template"].format(
            case_id="DIAG-Q35-01",
            contenido=contenido
        )
    }
]

# Enviar solicitud exclusivamente al servidor local
peticion = Request(
    configuracion["api_url"],
    data=json.dumps(
        solicitud_q35, ensure_ascii=False
    ).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST"
)

inicio = time.perf_counter()

with cliente_local.open(peticion, timeout=240) as conexion:
    respuesta_q35 = json.load(conexion)

tiempo_cliente_q35 = time.perf_counter() - inicio

# Mostrar la respuesta original
print("MODELO:", respuesta_q35["model"])
print("\nRESPUESTA:\n")
print(respuesta_q35.get("message", {}).get("content", ""))

print("\nMÉTRICAS:")
print("Tiempo cliente:", round(tiempo_cliente_q35, 3), "s")

for campo in (
    "total_duration",
    "load_duration",
    "prompt_eval_duration",
    "eval_duration"
):
    valor = respuesta_q35.get(campo)
    if valor is not None:
        print(campo, round(valor / 1e9, 3), "s")

for campo in (
    "prompt_eval_count",
    "prompt_eval_cached_count",
    "eval_count",
    "done_reason"
):
    print(campo, respuesta_q35.get(campo))

print(
    "Razonamiento devuelto:",
    bool(respuesta_q35.get("message", {}).get("thinking"))
)


MODELO: qwen3.5:9b-q4_K_M

RESPUESTA:

**Evidencia**
La base para esta justificación se fundamenta en los registros documentados: [E1] indica que el Autor X ha publicado un artículo sobre bases de datos académicas, y [E2] establece que el Autor Y ha publicado un artículo sobre recuperación de información. No existen datos disponibles que confirmen una colaboración previa entre ambos autores ni detalles específicos sobre el modelo predictor utilizado para generar la puntuación favorable.

**Interpretación**
Aunque las áreas de estudio mencionadas en [E1] y [E2] presentan una relación temática estrecha (bases de datos y recuperación de información), la similitud de temas por sí sola no demuestra que los autores hayan trabajado juntos ni colaboren en el futuro. La predicción recibida como "favorable" se mantiene como un resultado externo cuya causa específica no puede atribuirse a las publicaciones individuales descritas en [E1] y [E2], dado que la evidencia no documenta cómo estas influe

In [3]:
import json
import hashlib
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

MODELO = "qwen3.5:9b-q4_K_M"
API_LOCAL = "http://127.0.0.1:11434"
ARCHIVO_RESULTADOS = "resultados.jsonl"

# 1. Leer y verificar los archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

assert [c["case_id"] for c in casos] == [
    "S-005", "S-006", "S-007",
    "S-008", "S-009", "S-010"
]

assert config["api_url"] == API_LOCAL + "/api/chat"

# 2. Verificar el modelo local
cliente = build_opener(ProxyHandler({}))

with cliente.open(API_LOCAL + "/api/tags", timeout=10) as r:
    modelos = json.load(r)["models"]

modelo_info = next(m for m in modelos if m["name"] == MODELO)

with cliente.open(API_LOCAL + "/api/version", timeout=10) as r:
    version_ollama = json.load(r)["version"]

# 3. Recuperar resultados anteriores, si existen
try:
    with open(ARCHIVO_RESULTADOS, encoding="utf-8") as f:
        anteriores = [json.loads(line) for line in f if line.strip()]
except FileNotFoundError:
    anteriores = []

# Evitar mezclar resultados de modelos diferentes
for r in anteriores:
    assert r.get("solicitud", {}).get("model") == MODELO

registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_directa_v1"
    and r.get("grupo") == "comparacion"
}

# 4. Guardar todo en un único archivo
with open(ARCHIVO_RESULTADOS, "a", encoding="utf-8") as archivo:

    # Guardar el diagnóstico previo si sigue en memoria
    if (
        "DIAG-Q35-01" not in {r["case_id"] for r in anteriores}
        and all(x in globals() for x in (
            "solicitud_q35", "respuesta_q35", "tiempo_cliente_q35"
        ))
    ):
        assert solicitud_q35["model"] == MODELO
        assert solicitud_q35["think"] is False

        diagnostico = {
            "execution_id": str(uuid4()),
            "case_id": "DIAG-Q35-01",
            "grupo": "diagnostico",
            "configuracion_modelo": "qwen35_directa_v1",
            "modelo_digest": modelo_info["digest"],
            "ollama_version": version_ollama,
            "fecha_registro_utc": datetime.now(timezone.utc).isoformat(),
            "solicitud": solicitud_q35,
            "respuesta_original": respuesta_q35,
            "tiempo_cliente_s": round(tiempo_cliente_q35, 3)
        }

        archivo.write(json.dumps(diagnostico, ensure_ascii=False) + "\n")
        archivo.flush()
        print("DIAG-Q35-01 | diagnóstico guardado")

    # 5. Ejecutar los seis casos
    for caso in casos:

        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        solicitud = copy.deepcopy(config["request_defaults"])

        solicitud["model"] = MODELO
        solicitud["think"] = False

        # Misma configuración que el diagnóstico
        solicitud["options"].update({
            "num_ctx": 4096,
            "num_predict": 2048,
            "temperature": 0.7,
            "top_p": 0.8,
            "top_k": 20,
            "min_p": 0.0,
            "presence_penalty": 1.5,
            "repeat_penalty": 1.0
        })

        solicitud["messages"] = [
            {
                "role": "system",
                "content": config["system_prompt"]
            },
            {
                "role": "user",
                "content": config["user_prompt_template"].format(
                    case_id=case_id,
                    contenido=caso["contenido"]
                )
            }
        ]

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "tipo": caso["tipo"],
            "grupo": "comparacion",
            "configuracion_modelo": "qwen35_directa_v1",
            "prompt_version": "V3",
            "modelo_digest": modelo_info["digest"],
            "modelo_detalles": modelo_info.get("details"),
            "ollama_version": version_ollama,
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=240) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif respuesta.get("message", {}).get("thinking"):
                registro["estado"] = "revisar_modo"
            elif (
                respuesta.get("done_reason") == "stop"
                and respuesta.get("message", {}).get("content", "").strip()
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )
        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Guardar incluso errores y truncamientos
        archivo.write(json.dumps(registro, ensure_ascii=False) + "\n")
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO_RESULTADOS)


DIAG-Q35-01 | diagnóstico guardado
S-005 | ok | 521 tokens | 9.549 s
S-006 | ok | 508 tokens | 8.33 s
S-007 | ok | 629 tokens | 10.445 s
S-008 | ok | 555 tokens | 9.207 s
S-009 | ok | 536 tokens | 8.985 s
S-010 | ok | 504 tokens | 8.399 s

Archivo actualizado: resultados.jsonl


In [4]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

# Recuperar las ejecuciones existentes
with open("resultados.jsonl", encoding="utf-8") as f:
    registros = [json.loads(line) for line in f if line.strip()]

# Evitar repetir accidentalmente el diagnóstico
assert not any(
    r["case_id"] == "DIAG-Q35-02" for r in registros
), "El diagnóstico con razonamiento ya está registrado."

original = next(
    r for r in registros if r["case_id"] == "DIAG-Q35-01"
)

modelo = "qwen3.5:9b-q4_K_M"
assert original["solicitud"]["model"] == modelo
assert original["solicitud"]["think"] is False

# Conservar exactamente el prompt y la evidencia anteriores
solicitud = copy.deepcopy(original["solicitud"])

solicitud["think"] = True
solicitud["options"].update({
    "num_ctx": 8192,
    "num_predict": 4096,
    "temperature": 1.0,
    "top_p": 0.95
})

cliente = build_opener(ProxyHandler({}))
api = "http://127.0.0.1:11434"

# Verificar que seguimos usando exactamente el mismo modelo
with cliente.open(api + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_actual = next(m for m in instalados if m["name"] == modelo)
assert modelo_actual["digest"] == original["modelo_digest"]

registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-Q35-02",
    "grupo": "diagnostico",
    "configuracion_modelo": "qwen35_thinking_v1",
    "referencia": "DIAG-Q35-01",
    "modelo_digest": modelo_actual["digest"],
    "ollama_version": original["ollama_version"],
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

inicio = time.perf_counter()

try:
    peticion = Request(
        api + "/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif (
        respuesta.get("done_reason") == "stop"
        and respuesta.get("message", {}).get("content")
        and respuesta.get("message", {}).get("thinking")
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Añadir al único archivo de resultados
with open("resultados.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r["message"].get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r["message"].get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: truncada
Tiempo cliente: 86.367 s
Tokens entrada: 371
Tokens generados: 4096
Terminación: length
Razonamiento separado: True

RESPUESTA FINAL:

**Evidencia**
La justificación se basa en la documentación proporcionada sobre los perfiles académicos de los autores. [E1] establece que el Autor X ha publicado un artículo sobre bases de datos académicas y [E2] establece que el Autor Y ha publicado un artículo sobre recuperación de información. No existen datos adicionales sobre historial de colaboración, ni información específica sobre el modelo predictor responsable del resultado favorable recibido.

**Interpretación**
Existe una coincidencia temática en los documentos citados: ambos autores operan en el dominio de la ciencia de datos e investigación bibliográfica. Esta alineación académica proporciona un contexto semántico relevante que justifica la pertinencia de considerar una relación de coautoría, ya que las áreas investigadas comparten fundamentos disciplinares similares. No o

In [5]:
import json
import copy
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

# Recuperar el diagnóstico anterior
with open("resultados.jsonl", encoding="utf-8") as f:
    registros = [json.loads(line) for line in f if line.strip()]

assert not any(
    r["case_id"] == "DIAG-Q35-03" for r in registros
), "DIAG-Q35-03 ya existe."

anterior = next(
    r for r in registros if r["case_id"] == "DIAG-Q35-02"
)

assert anterior["estado"] == "truncada"
assert anterior["respuesta_original"]["done_reason"] == "length"

# Conservar la misma solicitud
solicitud = copy.deepcopy(anterior["solicitud"])

assert solicitud["model"] == "qwen3.5:9b-q4_K_M"
assert solicitud["think"] is True
assert solicitud["options"]["num_ctx"] == 8192
assert solicitud["options"]["num_predict"] == 4096

# Único cambio: ampliar el límite de salida
solicitud["options"]["num_predict"] = 6144

registro = {
    "execution_id": str(uuid4()),
    "case_id": "DIAG-Q35-03",
    "grupo": "diagnostico",
    "configuracion_modelo": "qwen35_thinking_6144",
    "referencia": "DIAG-Q35-02",
    "motivo": "Respuesta truncada con 4096 tokens",
    "modelo_digest": anterior["modelo_digest"],
    "ollama_version": anterior["ollama_version"],
    "fecha_inicio_utc": datetime.now(timezone.utc).isoformat(),
    "solicitud": solicitud,
    "reintentos": 0
}

cliente = build_opener(ProxyHandler({}))
inicio = time.perf_counter()

try:
    peticion = Request(
        "http://127.0.0.1:11434/api/chat",
        data=json.dumps(
            solicitud, ensure_ascii=False
        ).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST"
    )

    with cliente.open(peticion, timeout=300) as conexion:
        respuesta = json.load(conexion)

    registro["respuesta_original"] = respuesta

    if respuesta.get("done_reason") == "length":
        registro["estado"] = "truncada"
    elif (
        respuesta.get("done_reason") == "stop"
        and respuesta.get("message", {}).get("content")
        and respuesta.get("message", {}).get("thinking")
    ):
        registro["estado"] = "ok"
    else:
        registro["estado"] = "revisar"

except Exception as error:
    registro["estado"] = "error"
    registro["error_tipo"] = type(error).__name__
    registro["error"] = str(error)

registro["tiempo_cliente_s"] = round(
    time.perf_counter() - inicio, 3
)

registro["fecha_fin_utc"] = datetime.now(timezone.utc).isoformat()

# Guardar incluso si se vuelve a truncar
with open("resultados.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(registro, ensure_ascii=False) + "\n")

print("Estado:", registro["estado"])
print("Tiempo cliente:", registro["tiempo_cliente_s"], "s")

if "respuesta_original" in registro:
    r = registro["respuesta_original"]

    print("Tokens entrada:", r.get("prompt_eval_count"))
    print("Tokens generados:", r.get("eval_count"))
    print("Terminación:", r.get("done_reason"))
    print("Razonamiento separado:", bool(
        r["message"].get("thinking")
    ))

    print("\nRESPUESTA FINAL:\n")
    print(r["message"].get("content", ""))
else:
    print("Error:", registro.get("error"))

print("\nRegistro añadido a resultados.jsonl")


Estado: ok
Tiempo cliente: 85.167 s
Tokens entrada: 371
Tokens generados: 4633
Terminación: stop
Razonamiento separado: True

RESPUESTA FINAL:

Evidencia
El registro de datos disponible indica que el Autor X ha publicado un artículo sobre bases de datos académicas [E1] y que el Autor Y posee un artículo sobre recuperación de información [E2]. No se proporcionan datos de colaboración previa, afiliaciones institucionales compartidas ni métricas de interacción entre los investigadores.

Interpretación
Se distingue claramente la predicción recibida, favorable, del respaldo académico que emana de las publicaciones individuales. Las áreas temáticas cubiertas en [E1] y [E2] sugieren una coherencia disciplinaria que respalda la pertinencia académica de una colaboración sugerida entre los dos autores. Sin embargo, es crucial no atribuir la predicción a estos factores temáticos como causas documentadas del predictor, sino reconocer que la evidencia valida la viabilidad intelectual dentro del con

In [6]:
import json
import copy
import hashlib
import time
from uuid import uuid4
from datetime import datetime, timezone
from urllib.request import Request, build_opener, ProxyHandler

ARCHIVO = "resultados.jsonl"
MODELO = "qwen3.5:9b-q4_K_M"

# 1. Leer archivos compartidos
with open("../casos_comparacion_v1.json", "rb") as f:
    bytes_casos = f.read()

with open("../configuracion_v3.json", "rb") as f:
    bytes_config = f.read()

casos = json.loads(bytes_casos)
config = json.loads(bytes_config)

sha_casos = hashlib.sha256(bytes_casos).hexdigest()
sha_config = hashlib.sha256(bytes_config).hexdigest()

assert sha_casos == config["cases_sha256"]
assert sha_config == (
    "1c23edad394c7f555f05f2a464440e80cefa1023cb9026593e39eeebacd918af"
)

# 2. Recuperar resultados anteriores
with open(ARCHIVO, encoding="utf-8") as f:
    anteriores = [json.loads(line) for line in f if line.strip()]

directos = {
    r["case_id"]: r
    for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_directa_v1"
    and r.get("grupo") == "comparacion"
}

assert len(directos) == 6
assert set(directos) == {c["case_id"] for c in casos}

# Recuperar configuración del diagnóstico exitoso
diagnostico = next(
    r for r in anteriores
    if r["case_id"] == "DIAG-Q35-03"
)

assert diagnostico["estado"] == "ok"
assert diagnostico["solicitud"]["think"] is True
assert diagnostico["respuesta_original"]["done_reason"] == "stop"

opciones_thinking = copy.deepcopy(
    diagnostico["solicitud"]["options"]
)

# 3. Verificar modelo local
cliente = build_opener(ProxyHandler({}))
api = "http://127.0.0.1:11434"

with cliente.open(api + "/api/tags", timeout=10) as f:
    instalados = json.load(f)["models"]

modelo_actual = next(
    m for m in instalados if m["name"] == MODELO
)

assert modelo_actual["digest"] == diagnostico["modelo_digest"]

# Evitar duplicados
registrados = {
    r["case_id"] for r in anteriores
    if r.get("configuracion_modelo") == "qwen35_thinking_6144"
    and r.get("grupo") == "comparacion"
}

# 4. Ejecutar los seis casos
with open(ARCHIVO, "a", encoding="utf-8") as archivo:

    for caso in casos:
        case_id = caso["case_id"]

        if case_id in registrados:
            print(case_id, "| ya registrado")
            continue

        # Recuperar exactamente el prompt del caso original
        solicitud = copy.deepcopy(
            directos[case_id]["solicitud"]
        )

        assert solicitud["messages"][0]["content"] == (
            config["system_prompt"]
        )

        assert solicitud["messages"][1]["content"] == (
            config["user_prompt_template"].format(
                case_id=case_id,
                contenido=caso["contenido"]
            )
        )

        # Utilizar la configuración exitosa de DIAG-Q35-03
        solicitud["think"] = True
        solicitud["options"] = copy.deepcopy(opciones_thinking)

        registro = {
            "execution_id": str(uuid4()),
            "case_id": case_id,
            "grupo": "comparacion",
            "configuracion_modelo": "qwen35_thinking_6144",
            "prompt_version": "V3",
            "modelo_digest": modelo_actual["digest"],
            "ollama_version": diagnostico["ollama_version"],
            "sha256_casos": sha_casos,
            "sha256_configuracion": sha_config,
            "fecha_inicio_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "solicitud": solicitud,
            "reintentos": 0
        }

        inicio = time.perf_counter()

        try:
            peticion = Request(
                config["api_url"],
                data=json.dumps(
                    solicitud, ensure_ascii=False
                ).encode("utf-8"),
                headers={"Content-Type": "application/json"},
                method="POST"
            )

            with cliente.open(peticion, timeout=360) as conexion:
                respuesta = json.load(conexion)

            registro["respuesta_original"] = respuesta

            mensaje = respuesta.get("message", {})

            if respuesta.get("done_reason") == "length":
                registro["estado"] = "truncada"
            elif (
                respuesta.get("done_reason") == "stop"
                and mensaje.get("content")
                and mensaje.get("thinking")
            ):
                registro["estado"] = "ok"
            else:
                registro["estado"] = "revisar"

        except Exception as error:
            registro["estado"] = "error"
            registro["error_tipo"] = type(error).__name__
            registro["error"] = str(error)

        registro["tiempo_cliente_s"] = round(
            time.perf_counter() - inicio, 3
        )

        registro["fecha_fin_utc"] = datetime.now(
            timezone.utc
        ).isoformat()

        # Conservar incluso errores o truncamientos
        archivo.write(
            json.dumps(registro, ensure_ascii=False) + "\n"
        )
        archivo.flush()

        r = registro.get("respuesta_original", {})

        print(
            case_id,
            "|", registro["estado"],
            "|", r.get("eval_count", "-"), "tokens",
            "|", registro["tiempo_cliente_s"], "s"
        )

print("\nArchivo actualizado:", ARCHIVO)


S-005 | ok | 4635 tokens | 92.172 s
S-006 | ok | 4595 tokens | 83.905 s
S-007 | ok | 5571 tokens | 89.059 s
S-008 | ok | 4926 tokens | 78.193 s
S-009 | ok | 4343 tokens | 69.016 s
S-010 | ok | 4741 tokens | 75.237 s

Archivo actualizado: resultados.jsonl
